# Notebook to print a table of hyperparameters from YAML config files for different models and datasets.

In [1]:
import os
import yaml
import pandas as pd
import argparse

BASE_DIR = "../configs"


def load_yaml(path):
    """Load a YAML file safely."""
    if not os.path.exists(path):
        return None
    with open(path, "r") as f:
        return yaml.safe_load(f)


def extract_param(task, dataset, model, param_key):
    """Extract a specific parameter value from a config YAML."""
    yaml_path = os.path.join(BASE_DIR, task, dataset, f"{model}.yaml")
    data = load_yaml(yaml_path)['parameters']
    if data is None:
        return "FILE MISSING"
    return data.get(param_key, "KEY MISSING")


def build_table(key, task, datasets, models):
    """Create a table (DataFrame) of parameter values."""
    rows = []
    for model in models:
        row = {"model": model}
        for dataset in datasets:
            value = extract_param(task, dataset, model, key)
            row[dataset] = value
        rows.append(row)

    df = pd.DataFrame(rows)
    return df


In [2]:
# specify task, datasets, models, and parameter to extract
task = "lp"  # "nc" or "lp"
param = "model.hidden_dim"  # e.g., "model.hidden_dim", "training

if task == "lp":
    datasets = ["disease_lp", "airport", "cora", "citeseer", "pubmed"]
else:
    datasets = ["disease_nc", "airport", "cora", "citeseer", "pubmed"]
models = ["mlp", "gcn", "gat", "hybonet-lorentz", "hybonet-euclidean", "hgcn"]
df = build_table(param, task, datasets, models)
print(df.to_string(index=False))

            model  disease_lp  airport  cora  citeseer  pubmed
              mlp          24       32    64        32      64
              gcn          24       32    32       128      32
              gat         128       24    24        32      64
  hybonet-lorentz          16       16    16        32      16
hybonet-euclidean          16       32    32        32      16
             hgcn          64       24    64        32      64
